# Embedding Models Demo

Compare embedding models for RAG: quality, latency, and cost trade-offs.

**Models covered:**
- Local: all-MiniLM-L6-v2, BGE-small-en
- Ollama: nomic-embed-text

**Prerequisites:**
```bash
pip install sentence-transformers numpy
ollama pull nomic-embed-text
```

In [1]:
import time
import numpy as np
import requests
from sentence_transformers import SentenceTransformer

# Test corpus
DOCUMENTS = [
    "Remote work policy allows employees to work from home up to 3 days per week.",
    "Annual leave entitlement is 25 days for full-time employees.",
    "The company provides ergonomic equipment for home office setup.",
    "Performance reviews are conducted quarterly with manager feedback.",
    "IT support is available 24/7 for critical system issues.",
]

QUERIES = [
    "What are the WFH rules?",
    "How many vacation days do I get?",
    "Does the company help with home office equipment?",
]

print(f"Corpus: {len(DOCUMENTS)} docs, {len(QUERIES)} test queries")

Corpus: 5 docs, 3 test queries


In [2]:
# Load local models
print("Loading models...")
models = {
    "all-MiniLM-L6-v2": SentenceTransformer("all-MiniLM-L6-v2"),
    "bge-small-en": SentenceTransformer("BAAI/bge-small-en-v1.5"),
}
print("✓ Local models loaded")

def ollama_embed(text: str) -> list[float]:
    response = requests.post(
        "http://localhost:11434/api/embeddings",
        json={"model": "nomic-embed-text", "prompt": text}
    )
    return response.json()["embedding"]

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

Loading models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Local models loaded


In [3]:
def benchmark_model(name: str, encode_fn, queries: list, docs: list):
    """Benchmark embedding model on retrieval task."""
    # Embed documents
    start = time.time()
    doc_embs = [encode_fn(d) for d in docs]
    doc_time = (time.time() - start) * 1000 / len(docs)
    
    # Embed queries and search
    results = []
    for query in queries:
        start = time.time()
        q_emb = encode_fn(query)
        query_time = (time.time() - start) * 1000
        
        scores = [cosine_sim(q_emb, d) for d in doc_embs]
        top_idx = np.argmax(scores)
        results.append((query, top_idx, scores[top_idx], query_time))
    
    return {
        "model": name,
        "dim": len(doc_embs[0]),
        "doc_embed_ms": doc_time,
        "query_embed_ms": np.mean([r[3] for r in results]),
        "results": results
    }

# Run benchmarks
print("\nBenchmarking Models...")
print("=" * 60)

benchmarks = []

# Local models
for name, model in models.items():
    result = benchmark_model(name, lambda t: model.encode(t).tolist(), QUERIES, DOCUMENTS)
    benchmarks.append(result)
    print(f"✓ {name}: {result['dim']}d, {result['query_embed_ms']:.1f}ms/query")

# Ollama model
try:
    result = benchmark_model("nomic-embed-text", ollama_embed, QUERIES, DOCUMENTS)
    benchmarks.append(result)
    print(f"✓ nomic-embed-text: {result['dim']}d, {result['query_embed_ms']:.1f}ms/query")
except Exception as e:
    print(f"✗ nomic-embed-text: {e}")


Benchmarking Models...
✓ all-MiniLM-L6-v2: 384d, 12.2ms/query
✓ bge-small-en: 384d, 7.0ms/query
✓ nomic-embed-text: 768d, 27.8ms/query


In [4]:
# Results comparison
print("\nRetrieval Results by Model")
print("=" * 60)

for i, query in enumerate(QUERIES):
    print(f"\nQuery: \"{query}\"")
    for b in benchmarks:
        _, top_idx, score, _ = b["results"][i]
        print(f"  {b['model']}: doc{top_idx+1} (score: {score:.3f})")
        
print("\n\nModel Comparison Summary")
print("=" * 60)
print(f"{'Model':<25} {'Dim':<8} {'Query (ms)':<12} {'Cost':<10}")
print("-" * 60)
for b in benchmarks:
    cost = "Free" if "ollama" in b["model"].lower() or b["model"].startswith("all-") or b["model"].startswith("bge") else "API"
    print(f"{b['model']:<25} {b['dim']:<8} {b['query_embed_ms']:<12.1f} {cost:<10}")


Retrieval Results by Model

Query: "What are the WFH rules?"
  all-MiniLM-L6-v2: doc1 (score: 0.168)
  bge-small-en: doc1 (score: 0.494)
  nomic-embed-text: doc4 (score: 0.522)

Query: "How many vacation days do I get?"
  all-MiniLM-L6-v2: doc2 (score: 0.463)
  bge-small-en: doc1 (score: 0.602)
  nomic-embed-text: doc2 (score: 0.613)

Query: "Does the company help with home office equipment?"
  all-MiniLM-L6-v2: doc3 (score: 0.729)
  bge-small-en: doc3 (score: 0.849)
  nomic-embed-text: doc3 (score: 0.875)


Model Comparison Summary
Model                     Dim      Query (ms)   Cost      
------------------------------------------------------------
all-MiniLM-L6-v2          384      12.2         Free      
bge-small-en              384      7.0          Free      
nomic-embed-text          768      27.8         API       


---

## Recommendations

| Model | Quality | Latency | Cost | Best For |
|-------|---------|---------|------|----------|
| all-MiniLM-L6-v2 | Good | Very Fast | Free | Prototyping, latency-critical |
| BGE-small-en | Very Good | Fast | Free | Production (self-hosted) |
| nomic-embed-text | Very Good | Medium | Free | Ollama users, long context |
| text-embedding-3-small | Excellent | Medium | API | Production (API OK) |

**Production recommendation:** Start with BGE-small for self-hosted, OpenAI for API.